In [113]:
def get_chromaticities(var, i,tmpr):
    match i:
        case 0: mode = "natural_chrom"
        case 1:  mode = "chrom_correction"
        case 2:  mode = "spin_coherence"
    chrom_code = f"""
        INCLUDE '{var[0]}';
PROCEDURE MAIN;
VARIABLE WHERE 100; VARIABLE MRKR 100; VARIABLE GAMMA 1; VARIABLE MAPSE 1; VARIABLE MU 800; VARIABLE NBAR 800 3; VARIABLE EL 1;
VARIABLE MU0 1; VARIABLE PNUM 1; VARIABLE PSI0_DEG 1; VARIABLE NTURN 1; VARIABLE TUNE 800 3; VARIABLE MU_TP 800 3; VARIABLE CHROM_X 1; VARIABLE CHROM_Y 1;
VARIABLE SGx1 1; VARIABLE SGy1 1; VARIABLE EB1 1; VARIABLE SGx2 1; VARIABLE SGy2 1; VARIABLE A 100 2 ; VARIABLE B 100 2 ; VARIABLE G 100 2 ; VARIABLE R 100 2 ; VARIABLE F 100 6 ;
VARIABLE quadKx 1; VARIABLE quadKy 1; VARIABLE quadKz 1; VARIABLE OBJ 1;
      
GAMMA:={var[1]}; EB1 := {var[2]}; SGx1 :=  0; SGy1  := 0; SGx2 :=  0; SGy2  := 0;
    """ 
    match (mode):
        case ("natural_chrom"):
            chrom_code1 = f"""
                      OV 4 2 1;
                    SET_FOR_{var[3]}_CHROM GAMMA;
                  MRKR := 'chromaticities_{var[0]}_{mode}';
                  LATTICE SGx1 SGy1 SGx2 SGy2 EB1 1; PM 6;
                    TP MU_TP ;
                   WRITE 6 'DELTA DEPENDENT TUNES' MU_TP(1) MU_TP(2) ;
                    CHROM_X:= (MU_TP(1)|(0&0&0&0&1))*(1+1/GAMMA);
                    CHROM_Y:= (MU_TP(2)|(0&0&0&0&1))*(1+1/GAMMA);
    
                OV 3 3 0;
                SET_FOR_{var[3]} GAMMA;
                
                UM; LATTICE SGx1 SGy1 SGx2 SGy2 EB1 1;
                TSS MU NBAR 0;
                MU0 := CONS(MU);
                DAPEE MU 11 quadKx;
                DAPEE MU 33 quadKy;
                DAPEE MU 66 quadKz;
                OBJ := SQRT(SQR(quadKx) + SQR(quadKy)+SQR(quadKz));
    
                """
        case ("chrom_correction"):
            chrom_code1 = f"""
                      OV 4 2 1;
                SET_FOR_{var[3]}_CHROM GAMMA;
        
                FIT SGx1 SGy1 SGx2;
        
                LATTICE SGx1 SGy1 SGx2 SGy2 EB1 1;
                TP MU_TP;
                CHROM_X:= (MU_TP(1)|(0&0&0&0&1))*(1+1/GAMMA);
                CHROM_Y:= (MU_TP(2)|(0&0&0&0&1))*(1+1/GAMMA);
                OBJ:= SQRT(SQR(CHROM_X) + SQR(CHROM_Y));
        
                ENDFIT 1E-6 800 1 obj;
                MRKR := 'chrom_cor_{var[0]}_{mode}';
    
                OV 3 3 0;
                SET_FOR_{var[3]} GAMMA;
                
                UM; LATTICE SGx1 SGy1 SGx2 SGy2 EB1 1;
                TSS MU NBAR 0;
                MU0 := CONS(MU);
                DAPEE MU 11 quadKx;
                DAPEE MU 33 quadKy;
                DAPEE MU 66 quadKz;
                OBJ := SQRT(SQR(quadKx) + SQR(quadKy)+SQR(quadKz));
                """
        case ("spin_coherence"):
            chrom_code1 = f"""
                OV 3 3 0;
                SET_FOR_{var[3]} GAMMA;
                
                FIT SGx1 SGy1 SGx2 ;
                LATTICE SGx1 SGy1 SGx2 SGy2 EB1 1;
                TSS MU NBAR 0;
                MU0 := CONS(MU);
                DAPEE MU 11 quadKx;
                DAPEE MU 33 quadKy;
                DAPEE MU 66 quadKz;
                OBJ := SQRT(SQR(quadKx) + SQR(quadKy)+SQR(quadKz));
                ENDFIT 1E-9 1000 1 OBJ;
    
                OV 3 2 1 ; 
                    SET_FOR_{var[3]}_CHROM GAMMA ; 
                    UM ; LATTICE SGx1 SGy1 SGx2 SGy2 EB1 1 ; 
                    TP MU_TP ;
                    
                    CHROM_X := (MU_TP(1)|(0&0&0&0&1)) * (1+1/GAMMA) ;
                    CHROM_Y := (MU_TP(2)|(0&0&0&0&1)) * (1+1/GAMMA) ;
                """
                
    ending = f"""
                    OPENF 3618 '{tmpr}' 'REPLACE';
            
                       WRITE 3618 '#lattice 		  optimization 		  SGx1 		  SGx2 		  SGy1 		  chrom_x 		  chrom_y 		  quad_kx 		  quad_ky 		  quad_kz 		  obj';
                    	WRITE 3618 '{variables[0]} {mode} '&ST(SGx1)&' '&ST(SGx2)&' '&ST(SGy1)&' '&ST(chrom_x)&' '&ST(chrom_y)&' '&ST(quadkx)&' '&ST(quadky)&' '&ST(quadkz)&' '&ST(obj);
                    	CLOSEF 3618;
        
        ENDPROCEDURE;
        
        PROCEDURE RUN;
          MAIN;
        ENDPROCEDURE;
        RUN; END;
    """
    
    with open(f"py.fox", "w") as f:
        f.write(chrom_code)
        f.write(chrom_code1)
        f.write(ending)

In [114]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

"""------------INFO---------------------------------------------
variables = [lattice, gamma, EB1, particle, element_number, lattice_length]
 i: 0 - "natural_chrom"; 1 - "chrom_correction"; 2 - "spin_coherence"
директория "py.fox" совпадает с директорией данного jupyter файла
py.fox и py.dat обновляются после каждого запуска
ДЛЯ СБОРА ДАННЫХ В 1 ФАЙЛ последовательно ЗАПУСТИТЬ текущую клетку + 2 нижних
-------------------------------------------------------------------
"""
temporary_file = "C:/Users/palo4/Desktop/MEPHI/Master/HNP/MIPT conf/COSY/tracking data/py.dat"
chromaticities_data = "C:/Users/palo4/Desktop/MEPHI/Master/HNP/MIPT conf/COSY/chrom_optimization.dat"

variables = ["electrostatic", 1.24810736, 112.464392, "protons", 540, 419.10]
#variables = ["magnetic_2p", 1.143914, 0, "deuterons", 100, 141.014]

get_chromaticities(variables, 0, temporary_file)

In [115]:
import subprocess
c = "py.fox"
subprocess.Popen(f'start cmd /c cosy {c}', shell=True)

<Popen: returncode: None args: 'start cmd /c cosy py.fox'>

In [110]:
with open(f"{temporary_file}") as f:
    parts = f.readlines()[1].strip().split()

# форматы: первые два столбца — строки, остальные — числа
formats = [
    "{:<16}",     # lattice
    "{:<20}",     # optimization
    "{:<16.6e}",  # SGx1
    "{:<16.6e}",  # SGx2
    "{:<16.6e}",  # SGy1
    "{:<18.6e}",  # chrom_x
    "{:<18.6e}",  # chrom_y
    "{:<18.6e}",  # quad_kx
    "{:<18.6e}",  # quad_ky
    "{:<18.6e}",  # quad_kz
    "{:<18.6e}",  # obj
]

# типы: первые два — строки, остальные — числа
values = [parts[0], parts[1]] + [float(x) for x in parts[2:]]

# собираем строку с пробелом между колонками
formatted_line = " ".join(fmt.format(val) for fmt, val in zip(formats, values))

# заголовок
headers = ["lattice","optimization","SGx1","SGx2","SGy1","chrom_x","chrom_y","quad_kx","quad_ky","quad_kz","obj"]
widths  = [16,20,16,16,16,18,18,18,18,18,18]
header = " ".join(f"{h:<{w}}" for h, w in zip(headers, widths))

# запись
with open(f"{chromaticities_data}", "a+") as f:
    f.seek(0)
    existing = f.read()

    if header not in existing:
        f.write(header + "\n")

    if formatted_line not in existing:
        f.write(formatted_line + "\n")